In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [37]:
# Get feature name
featurenamefile = 'Feature_name.dat'
with open(featurenamefile) as file:
    feature_name = [line.rstrip() for line in file]
file.close()
ft_dict = {key: i for i, key in enumerate(feature_name)}
n_feature = len(feature_name)

In [38]:
# read all dataset
# df_normal
# df_coapdos
# df_mqttflood
# df_mqttslowdos
# df_pingflood
# df_portscan
# df_sqlmap
# df_tcpsyn
# df_coapdos = pd.read_csv('./coap_dos/coap_dos_10x1MB_NAS_Traffic_21_Apr_2024_06_58.csv',header=None, names=feature_name)
# df_mqttflood = pd.read_csv('./mqttPUBflood/mqttpub_flood_100x10MB_NAS_Traffic_21_Apr_2024_03_39.csv',header=None, names=feature_name)
# df_mqttslowdos = pd.read_csv('./mqttslowdos/mqtt_slowdos_2k_NAS_Traffic_21_Apr_2024_04_52.csv',header=None, names=feature_name)
# df_pingflood = pd.read_csv('./pingflood/ping_flood_NAS_Traffic_17_Apr_2024_18_42.csv',header=None, names=feature_name)
# df_portscan = pd.read_csv('./port_scan/portscan_NAS_Traffic_17_Apr_2024_16_50.csv',header=None, names=feature_name)
# df_sqlmap = pd.read_csv('./sqlmap/sqlmap_NAS_Traffic_17_Apr_2024_16_07.csv',header=None, names=feature_name)
# df_tcpsyn = pd.read_csv('./tcpflood/tcp_syn_NAS_Traffic_18_Apr_2024_14_47.csv',header=None, names=feature_name)
# df_normal = pd.read_csv('./normal/normal_NAS_Traffic_20_Apr_2024_16_00.csv',header=None, names=feature_name)
df_coapdos = pd.read_csv('./coap_dos/coap_dos_eval_NAS_Traffic_21_Apr_2024_16_24.csv',header=None, names=feature_name)
df_mqttflood = pd.read_csv('./mqttPUBflood/mqttpub_flood_100x100MB_NAS_Traffic_18_Apr_2024_15_42.csv',header=None, names=feature_name)
df_mqttslowdos = pd.read_csv('./mqttslowdos/mqtt_slowdos_1k_eval_NAS_Traffic_21_Apr_2024_04_45.csv',header=None, names=feature_name)
df_pingflood = pd.read_csv('./pingflood/pingflood_spoofed_eval_NAS_Traffic_21_Apr_2024_02_50.csv',header=None, names=feature_name)
df_portscan = pd.read_csv('./port_scan/portscan_10k_eval_NAS_Traffic_20_Apr_2024_21_57.csv',header=None, names=feature_name)
df_sqlmap = pd.read_csv('./sqlmap/sqlmap_eval_NAS_Traffic_20_Apr_2024_21_13.csv',header=None, names=feature_name)
df_tcpsyn = pd.read_csv('./tcpflood/tcpsyn_rand_src_eval_NAS_Traffic_20_Apr_2024_22_05.csv',header=None, names=feature_name)
df_normal = pd.read_csv('./normal/normal_eval_NAS_Traffic_21_Apr_2024_05_01.csv',header=None, names=feature_name)
df_normal['label'] = 0
df_coapdos['label'] = 1
df_mqttflood['label'] = 2
df_mqttslowdos['label'] = 3
df_pingflood['label'] = 4
df_portscan['label'] = 5
df_sqlmap['label'] = 6
df_tcpsyn['label'] = 7


In [39]:
# Drop columns
drop_columns = ['idx','level','session_id','ue_id','direction']
df_normal.drop(columns=drop_columns,inplace=True)
df_coapdos.drop(columns=drop_columns,inplace=True)
df_mqttflood.drop(columns=drop_columns,inplace=True)
df_mqttslowdos.drop(columns=drop_columns,inplace=True)
df_pingflood.drop(columns=drop_columns,inplace=True)
df_portscan.drop(columns=drop_columns,inplace=True)
df_sqlmap.drop(columns=drop_columns,inplace=True)
df_tcpsyn.drop(columns=drop_columns,inplace=True)

In [40]:
# zero time stamp
def zerotime(df):
    min_timestamp = df['timestamp'].min()
    df['timestamp']= df['timestamp'].apply(lambda x: x-min_timestamp)

def scale(df,ratio):
    df['timestamp']= df['timestamp'].apply(lambda x: x/ratio)

zerotime(df_normal)
zerotime(df_coapdos)
zerotime(df_mqttflood)
zerotime(df_mqttslowdos)
zerotime(df_pingflood)
zerotime(df_portscan)
scale(df_portscan,10)
zerotime(df_sqlmap)
zerotime(df_tcpsyn)

In [41]:
print(df_normal.shape)
print(df_coapdos.shape)
print(df_mqttflood.shape)
print(df_mqttslowdos.shape)
print(df_pingflood.shape)
print(df_portscan.shape)
print(df_sqlmap.shape)
print(df_tcpsyn.shape)

(140778, 3)
(37541, 3)
(16208, 3)
(11725, 3)
(31708, 3)
(20007, 3)
(6273, 3)
(49998, 3)


In [ ]:
# random new time stamp
def random_starttime(maindf:pd.DataFrame, smalldfs: (pd.DataFrame), min_starttime=None, max_starttime=None):
    '''
    merge small dataset in the big dataset intertwainly in random start time manner
    maindf : |--------------------------------------|
    smalldf:        |--------|     |----------|
    '''
    total_attack_time = 0
    for df in smalldfs:
        total_attack_time += df['timestamp'].max()
    print(total_attack_time/60000)

    total_time = maindf['timestamp'].iloc[-1]
    print(total_time/60000)

    n_split = len(smalldfs)
    chunk_time = total_time//n_split

    for i in range(0,len(smalldfs)):
        chunk_start_time = chunk_time*i
        attack_time = smalldfs[i]['timestamp'].iloc[-1]
        print(attack_time/60000)
        spare_time = chunk_time - attack_time
        if spare_time > 0:
            attack_start_time = chunk_start_time + np.random.randint(spare_time)
        else:
            attack_start_time = chunk_start_time

        smalldfs[i]['timestamp'] = smalldfs[i]['timestamp'].apply(lambda x: x + attack_start_time)
        
    df_chunks = [maindf]+smalldfs
    merged_df = pd.concat(df_chunks,ignore_index=True)
    merged_df.sort_values('timestamp',inplace=True)
    merged_df.reset_index(drop=True,inplace=True)
    
    return merged_df
    

## Spliting Training dataset


In [43]:
# split normal dataset into 7 chunks
TA_normal_dfs = np.array_split(df_normal, 7)
for df in TA_normal_dfs:
    zerotime(df)

# split df_coapdos into 3 chunks
coapdos_dfs = np.array_split(df_coapdos, 3)
for df in coapdos_dfs:
    zerotime(df)

# split df_mqttflood into 2 chunks
mqttflood_dfs = np.array_split(df_mqttflood, 2)
for df in mqttflood_dfs:
    zerotime(df)

# split df_mqttslowdos into 2 chunks
mqttslowdos_dfs = np.array_split(df_mqttslowdos, 2)
for df in mqttslowdos_dfs:
    zerotime(df)

# split df_pingflood into 3 chunks
pingflood_dfs = np.array_split(df_pingflood, 4)
for df in pingflood_dfs:
    zerotime(df)

# split df_portscan into 3 chunks
portscan_dfs = np.array_split(df_portscan, 4)
for df in portscan_dfs:
    zerotime(df)

# split df_sqlmap into 3 chunks
sqlmap_dfs = np.array_split(df_sqlmap, 2)
for df in sqlmap_dfs:
    zerotime(df)

# split df_tcpsyn into 3 chunks
tcpsyn_dfs = np.array_split(df_tcpsyn, 3)
for df in tcpsyn_dfs:
    zerotime(df)


/home/nlag/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [44]:
merged_dfs = []
# TA1 - pingflood, tcpsyn, coapdos
merged_df_TA1 = random_starttime(TA_normal_dfs[0], [pingflood_dfs[0],tcpsyn_dfs[0],coapdos_dfs[0]])
merged_df_TA1['packet_hex'] = merged_df_TA1['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA1)

# TA2 - portscan, coapdos 
merged_df_TA2 = random_starttime(TA_normal_dfs[1], [portscan_dfs[0],coapdos_dfs[1]])
merged_df_TA2['packet_hex'] = merged_df_TA2['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA2)

# TA3 - coapdos, mqttflood, mqttslowdos, tcpsyn
merged_df_TA3 = random_starttime(TA_normal_dfs[2], [coapdos_dfs[2],mqttflood_dfs[0],mqttslowdos_dfs[0],tcpsyn_dfs[1]])
merged_df_TA3['packet_hex'] = merged_df_TA3['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA3)

# TA4 - pingflood, mqttflood, mqttslowdos
merged_df_TA4 = random_starttime(TA_normal_dfs[3], [pingflood_dfs[1],mqttflood_dfs[1],mqttslowdos_dfs[1]])
merged_df_TA4['packet_hex'] = merged_df_TA4['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA4)

# TA5 - portscan, pingflood
merged_df_TA5 = random_starttime(TA_normal_dfs[4], [portscan_dfs[1],pingflood_dfs[2]])
merged_df_TA5['packet_hex'] = merged_df_TA5['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA5)

# TA6 - portscan, sqlmap, pingflood 
merged_df_TA6 = random_starttime(TA_normal_dfs[5], [portscan_dfs[2],sqlmap_dfs[0],pingflood_dfs[3]])
merged_df_TA6['packet_hex'] = merged_df_TA6['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA6)

# TA7 - tcpsyn, portscan, sqlmap
merged_df_TA7 = random_starttime(TA_normal_dfs[6], [tcpsyn_dfs[2],portscan_dfs[3],sqlmap_dfs[1]])
merged_df_TA7['packet_hex'] = merged_df_TA7['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA7)


4.840933333333333
9.53885
0.19825
2.7785
1.8641833333333333
0.8924516666666666
8.8824
0.04176833333333333
0.8506833333333333
4.939416666666666
8.985666666666667
0.5870833333333333
1.29
0.2833333333333333
2.779
1.6665
8.869483333333333
0.22158333333333333
1.1640833333333334
0.2808333333333333
0.264015
8.857333333333333
0.041931666666666666
0.22208333333333333
1.0192016666666666
8.942166666666667
0.04186833333333334
0.7835
0.19383333333333333
3.130035
16.113183333333332
2.779
0.04186833333333334
0.30916666666666665


## Spliting evaluation dataset


In [46]:
# (140778, 3)
# (37541, 3)
# (16208, 3)
# (11725, 3)
# (31708, 3)
# (20007, 3)
# (6273, 3)
# (49998, 3)

# split normal dataset into 7 chunks
TA_normal_dfs = np.array_split(df_normal, 7)
for df in TA_normal_dfs:
    zerotime(df)

# split df_coapdos into 2 chunks
coapdos_dfs = np.array_split(df_coapdos, 2)
for df in coapdos_dfs:
    zerotime(df)

# split df_mqttflood into 2 chunks
mqttflood_dfs = np.array_split(df_mqttflood, 2)
for df in mqttflood_dfs:
    zerotime(df)

# split df_mqttslowdos into 2 chunks
mqttslowdos_dfs = np.array_split(df_mqttslowdos, 2)
for df in mqttslowdos_dfs:
    zerotime(df)

# split df_pingflood into 2 chunks
pingflood_dfs = np.array_split(df_pingflood, 2)
for df in pingflood_dfs:
    zerotime(df)

# split df_portscan into 2 chunks
portscan_dfs = np.array_split(df_portscan, 2)
for df in portscan_dfs:
    zerotime(df)

# split df_sqlmap into 1 chunks
# sqlmap_dfs = np.array_split(df_sqlmap, 1)
sqlmap_dfs = [df_sqlmap]
for df in sqlmap_dfs:
    zerotime(df)

# split df_tcpsyn into 3 chunks
tcpsyn_dfs = np.array_split(df_tcpsyn, 3)
for df in tcpsyn_dfs:
    zerotime(df)


In [47]:
merged_dfs = []
# TA1 - portscan, tcpsyn
merged_df_TA1 = random_starttime(TA_normal_dfs[0], [portscan_dfs[0],tcpsyn_dfs[0]])
merged_df_TA1['packet_hex'] = merged_df_TA1['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA1)

# TA2 - portscan, tcpsyn
merged_df_TA2 = random_starttime(TA_normal_dfs[1], [portscan_dfs[1],tcpsyn_dfs[1]])
merged_df_TA2['packet_hex'] = merged_df_TA2['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA2)

# TA3 - coapdos, mqttslowdos
merged_df_TA3 = random_starttime(TA_normal_dfs[2], [coapdos_dfs[0],mqttslowdos_dfs[0]])
merged_df_TA3['packet_hex'] = merged_df_TA3['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA3)

# TA4 - coapdos, mqttflood 
merged_df_TA4 = random_starttime(TA_normal_dfs[3], [coapdos_dfs[1],mqttflood_dfs[0]])
merged_df_TA4['packet_hex'] = merged_df_TA4['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA4)

# TA5 - mqttflood
merged_df_TA5 = random_starttime(TA_normal_dfs[4], [mqttflood_dfs[1]])
merged_df_TA5['packet_hex'] = merged_df_TA5['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA5)

# TA6 - mqttslowdos, pingflood
merged_df_TA6 = random_starttime(TA_normal_dfs[5], [mqttslowdos_dfs[1],pingflood_dfs[0]])
merged_df_TA6['packet_hex'] = merged_df_TA6['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA6)

# TA7 - pingflood, sqlmap
merged_df_TA7 = random_starttime(TA_normal_dfs[6], [pingflood_dfs[1],sqlmap_dfs[0]])
merged_df_TA7['packet_hex'] = merged_df_TA7['packet_hex'].map(lambda x: x[2:])
merged_dfs.append(merged_df_TA7)


2.8622
9.53885
0.0837
2.7785
2.8627683333333334
8.8824
0.08376833333333333
2.779
2.6691666666666665
8.985666666666667
2.3858333333333333
0.2833333333333333
2.2063333333333333
8.869483333333333
0.9163333333333333
1.29
1.1640833333333334
8.857333333333333
1.1640833333333334
0.7006666666666667
8.942166666666667
0.2808333333333333
0.41983333333333334
1.5085833333333334
16.113183333333332
0.41591666666666666
1.0926666666666667


In [48]:
for df in merged_dfs:
    print(df['label'].value_counts())

label
0    20112
7    16666
5    10004
Name: count, dtype: int64
label
0    20111
7    16666
5    10003
Name: count, dtype: int64
label
0    20111
1    18771
3     5863
Name: count, dtype: int64
label
0    20111
1    18770
2     8104
Name: count, dtype: int64
label
0    20111
2     8104
Name: count, dtype: int64
label
0    20111
4    15854
3     5862
Name: count, dtype: int64
label
0    20111
4    15854
6     6273
Name: count, dtype: int64


In [49]:
for i, df in enumerate(merged_dfs):
    df.to_csv(f'merged_df_eval_TA{i+1}.csv', index=False)